# Ticket 6: account-level reconciliation (2024-01)

Issue #6 asks for two hypotheses tested before any transform: the
publishable net for P01 doesn't come out near zero, tens of millions off.
Testing that here before building `recon_period_summary`.

In [ ]:
import duckdb # type: ignore

con = duckdb.connect("../warehouse.duckdb", read_only=True)
SCOPE = "company_code = 1000 AND fiscal_year = 2024 AND fiscal_period = 1"

In [2]:
# H1: does the ledger balance on debit/credit across the whole scope?
# (the fundamental double-entry check, independent of local_amount)
con.execute(f"SELECT ROUND(SUM(debit_amount) - SUM(credit_amount), 2) FROM stg_gl WHERE {SCOPE}").fetchall()

[(90.0,)]

**90.0** - exactly the gap from the 1 known unbalanced document (ticket
5's `unbalanced_document` finding). H1 rejected: the ledger balances
correctly on debit/credit everywhere else. The problem isn't here.

In [3]:
# H2: which documents don't net to zero on local_amount, even though they balance on debit/credit?
con.execute(f"""
    WITH doc_net AS (
        SELECT document_id, document_type, ROUND(SUM(local_amount),2) net, COUNT(*) n_lines
        FROM stg_gl WHERE {SCOPE} GROUP BY 1, 2
    )
    SELECT COUNT(*) offending_docs, ROUND(SUM(net), 2) total_from_offenders
    FROM doc_net WHERE ABS(net) > 0.01
""").fetchall()

[(20, 97144587.1)]

**20 documents** (out of 3,518) account for **97,144,587.1**, effectively
the entire period net. H2 confirmed.

In [4]:
# What do these 20 documents have in common? Line count, fraud/anomaly flags, currency
con.execute(f"""
    SELECT document_id, document_type, COUNT(*) n_lines,
           ROUND(SUM(local_amount),2) net, bool_or(is_fraud) any_fraud, bool_or(is_anomaly) any_anomaly
    FROM stg_gl WHERE {SCOPE}
    GROUP BY 1, 2 HAVING ABS(ROUND(SUM(local_amount),2)) > 0.01
    ORDER BY ABS(net) DESC
""").fetchall()

[('0a121d1e-7334-81f7-2608-9bb9a6e55fab', 'KR', 37, 41646671.85, False, False),
 ('ff37e2f8-570f-80bb-046a-739063b0c380', 'KR', 64, 16496633.76, False, False),
 ('baf325f1-de3b-8184-2f67-00817c86f1d7', 'SA', 79, 11304573.28, False, False),
 ('51ad9376-2306-8f69-0bee-722d35801425', 'KR', 62, 7629389.71, False, False),
 ('bd3fc32d-5d1d-8922-3373-87b930724279', 'SA', 55, 5932511.0, False, False),
 ('dfc53943-7e93-843c-2664-683f0e4c28c9', 'DR', 71, 4844919.41, False, True),
 ('20a0d33e-f594-8582-0eee-9da0e065bc85', 'DR', 33, 2044140.0, False, False),
 ('dbbe683e-696e-8d08-2d71-67834f4954b0', 'KR', 53, 1356524.52, False, False),
 ('2262522b-2e0f-87a8-2304-a1881e4a55aa', 'SA', 33, 1305988.15, False, False),
 ('cbf097de-1be0-8976-2aab-d40b3316783a', 'HR', 60, 1130060.98, False, False),
 ('c5dd30e6-02e8-8a81-28d1-a4fe713550f2', 'SA', 44, 1128214.63, False, False),
 ('fdc7e1c3-e307-87cb-2d81-3dbb83595714', 'SA', 56, 1104546.24, False, False),
 ('7774e114-9b83-8201-3ccf-6de980be4e44', 'KR', 53, 

Every one of the 20 has an unusually high line count (18 to 81 lines,
versus 2-4 for a typical document). Only 2 of the 20 carry an
`is_fraud`/`is_anomaly` flag, so the pattern doesn't track those columns.
Line count is the common thread. Next: look inside one.

In [5]:
# Line-level detail of the single biggest offender
con.execute("""
    SELECT line_number, debit_amount, credit_amount, local_amount, currency, exchange_rate
    FROM stg_gl WHERE document_id = '0a121d1e-7334-81f7-2608-9bb9a6e55fab' ORDER BY line_number
""").fetchall()

[(1, 20542.31, 0.0, 1189904.91, 'USD', 1.0),
 (2, 26214.18, 0.0, 1189904.91, 'USD', 1.0),
 (3, 39496.41, 0.0, 1189904.91, 'USD', 1.0),
 (4, 19826.91, 0.0, 1189904.91, 'USD', 1.0),
 (5, 41252.89, 0.0, 1189904.91, 'USD', 1.0),
 (6, 35607.72, 0.0, 1189904.91, 'USD', 1.0),
 (7, 31917.4, 0.0, 1189904.91, 'USD', 1.0),
 (8, 44913.33, 0.0, 1189904.91, 'USD', 1.0),
 (9, 20703.65, 0.0, 1189904.91, 'USD', 1.0),
 (10, 43786.99, 0.0, 1189904.91, 'USD', 1.0),
 (11, 21992.95, 0.0, 1189904.91, 'USD', 1.0),
 (12, 40275.02, 0.0, 1189904.91, 'USD', 1.0),
 (13, 41081.14, 0.0, 1189904.91, 'USD', 1.0),
 (14, 27261.88, 0.0, 1189904.91, 'USD', 1.0),
 (15, 23448.97, 0.0, 1189904.91, 'USD', 1.0),
 (16, 41664.68, 0.0, 1189904.91, 'USD', 1.0),
 (17, 26002.33, 0.0, 1189904.91, 'USD', 1.0),
 (18, 42966.98, 0.0, 1189904.91, 'USD', 1.0),
 (19, 29531.64, 0.0, 1189904.91, 'USD', 1.0),
 (20, 38385.24, 0.0, 1189904.91, 'USD', 1.0),
 (21, 34411.79, 0.0, 1189904.91, 'USD', 1.0),
 (22, 46851.54, 0.0, 1189904.91, 'USD', 1.0)

Every line has a different `debit_amount` (20,542.31, 26,214.18,
39,496.41, ...) but the exact same `local_amount`: **1,189,904.91**,
repeated on every one of the 37 lines, in a `USD`/`exchange_rate=1.0`
document where `local_amount` should just equal `debit_amount` per line
(as it correctly does on ordinary 2-line documents, checked separately).
This reads as a document-level total broadcast onto every line instead
of a real per-line amount, likely an artifact of how the source dataset
was generated for documents with many lines, not a rounding issue.

In [6]:
# Do these 20 documents overlap with ticket 5's already-logged local_amount_imbalance finding?
con.execute("SELECT COUNT(*) FROM dq_violations WHERE check_name = 'local_amount_imbalance'").fetchall()

[(23,)]

**23** - the same check already caught all 20 of these (plus 3 more at
$0.01-0.02, genuine rounding noise, not this defect). Nothing new is
being detected here; what's new is the materiality: `docs/definitions.md`
currently describes this cause as "(currency-conversion rounding)", which
is right for 3 of the 23 rows and wrong for the other 20.

In [7]:
# Given the defect is in stg_gl itself (not something #4/#5 introduced), how many gl_accounts
# actually show a stg-vs-fact gap once fact_gl_line is built from the same (defective) source?
con.execute(f"""
    WITH stg AS (SELECT gl_account, ROUND(SUM(local_amount),2) stg_total FROM stg_gl WHERE {SCOPE} GROUP BY 1),
         fact AS (SELECT gl_account, ROUND(SUM(local_amount),2) fact_total FROM fact_gl_line GROUP BY 1)
    SELECT COUNT(*) accounts,
           SUM(CASE WHEN ABS(COALESCE(stg_total,0)-COALESCE(fact_total,0))>0.01 THEN 1 ELSE 0 END) accounts_with_real_gap
    FROM stg FULL OUTER JOIN fact USING (gl_account)
""").fetchall()

[(502, 2)]

**502 accounts total, only 2 with a real stg-vs-fact gap** - both from the
1 document `fact_gl_line` correctly excludes for being unbalanced. The
account-level reconciliation this ticket builds will be almost entirely
clean: `fact_gl_line` inherits the same `local_amount` values `stg_gl`
has, defect included, because nothing in #4/#5 touches that column.

## What I've got

- **H1 (debit/credit imbalance): rejected.** The ledger balances
  everywhere except the 1 already-known unbalanced document.
- **H2 (local_amount data defect): confirmed.** 20 documents, all with
  unusually many lines, have `local_amount` repeated identically across
  every line instead of a real per-line value. They account for the
  entire ~97.1M period net. Already caught by ticket 5's
  `local_amount_imbalance` check (23 rows total, 20 of them this defect,
  3 genuine rounding).
- **The stg-vs-fact reconciliation itself will be nearly trivial**: 500
  of 502 accounts match exactly, because `fact_gl_line` carries the same
  defective `local_amount` values `stg_gl` does. The interesting finding
  isn't a stg/fact mismatch, it's that the close total itself rests on
  ~$97M of `local_amount` that doesn't mean what it's supposed to mean.